In [6]:
from data import CleanNumericDatasets
from reservoirs import CPRC
from circuits import CPCircuit
from ELM import QuantumELM
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from utility import Evaluator

In [7]:
dim = 18
ds = CleanNumericDatasets(seed=3, scale=True)

X_train, X_test, y_train, y_test = ds.get(
    name="synthetic_classification_hard",
    sample_size=2000,
    feature_size=dim,
    return_split=True,
    test_size=0.2
)

ds.info(print_out=True)

name: synthetic_classification_hard
task: classification
source: synthetic
n_samples_requested: 2000
n_features_requested: 18
n_samples_returned: 2000
n_features_returned: 18
n_classes: 2
feature_transform: synthetic(hard)
scaler: standard
split: train_test_split
test_size: 0.2
random_state: 3
stratified: True
y_type: int64
X_dtype: float32
notes: Hard synthetic classification: low class_sep, label noise, class imbalance.


{'name': 'synthetic_classification_hard',
 'task': 'classification',
 'source': 'synthetic',
 'n_samples_requested': 2000,
 'n_features_requested': 18,
 'n_samples_returned': 2000,
 'n_features_returned': 18,
 'n_classes': 2,
 'feature_transform': 'synthetic(hard)',
 'scaler': 'standard',
 'split': 'train_test_split',
 'test_size': 0.2,
 'random_state': 3,
 'stratified': True,
 'y_type': 'int64',
 'X_dtype': 'float32',
 'notes': 'Hard synthetic classification: low class_sep, label noise, class imbalance.'}

In [8]:
# X_train, X_test, y_train, y_test = train_test_split(Xc, yc, test_size=0.2, random_state=42)

In [9]:
CP_params = [-np.pi/3, np.pi/6, -np.pi/9, np.pi/7, np.pi/9, -np.pi/7]
cprc = CPRC(dim=dim, execution_mode='simulation', CP_params=CP_params, kernel = True)

In [32]:
elm = QuantumELM(
        reservoir=cprc, 
        regularization=1, 
        show_progress=True, 
        model_type='ridge_classifier',  # Options: 'ridge', 'lasso', 'linear', 'svr', 'svc',  or "ridge_classifier","logreg", "svc" for classification
        )
elm.fit(X_train, y_train)

ELM feature extraction: 100%|██████████| 1600/1600 [01:13<00:00, 21.76 sample/s]


In [33]:
predictions = elm.predict(X_test)
rmse = mean_squared_error(y_test, predictions)
print("RMSE:", rmse)

ELM feature extraction: 100%|████████████| 400/400 [00:18<00:00, 21.99 sample/s]

RMSE: 0.3625


In [34]:
Evaluator(y_test, predictions).evaluate()

Evaluation Metrics:
Accuracy:  0.6375
Recall:  0.4205607476635514
F1 Score:  0.3829787234042553
Precision:  0.3515625
Matthews Correlation Coefficient:  0.13027381361458099

Confusion Matrix:
TN, FP, FN, TP
[210  83  62  45]


In [15]:
from benchmark import run_classical_benchmarks

In [16]:
results = run_classical_benchmarks(X_train, y_train, X_test, y_test, seed=3, cv_splits=5)
for k, v in results.items():
    print("\n", k)
    print(" best_params:", v["best_params"])
    print(" cv_best_balanced_acc:", v["cv_best_balanced_acc"])
    print(" test_metrics:", v["test_metrics"])


 logreg
 best_params: {'clf__C': 0.01}
 cv_best_balanced_acc: 0.6733258876318514
 test_metrics: {'accuracy': 0.67, 'balanced_accuracy': 0.6768524130011802, 'f1': 0.5285714285714286, 'precision': 0.4277456647398844, 'recall': 0.6915887850467289, 'roc_auc': 0.7156071576664221}

 svm_rbf
 best_params: {'clf__C': 1, 'clf__gamma': 0.1}
 cv_best_balanced_acc: 0.7959304427692238
 test_metrics: {'accuracy': 0.7975, 'balanced_accuracy': 0.7876144301617174, 'f1': 0.6693877551020408, 'precision': 0.5942028985507246, 'recall': 0.7663551401869159, 'roc_auc': 0.855507001371567}

 random_forest
 best_params: {'max_depth': None, 'max_features': None, 'min_samples_leaf': 5}
 cv_best_balanced_acc: 0.8044385212185077
 test_metrics: {'accuracy': 0.8375, 'balanced_accuracy': 0.8060189467640586, 'f1': 0.7085201793721974, 'precision': 0.6810344827586207, 'recall': 0.7383177570093458, 'roc_auc': 0.8811202194507353}

 grad_boost
 best_params: {'learning_rate': 0.2, 'max_depth': 3, 'n_estimators': 600}
 cv_bes